#### Imports and Load Data

In [ ]:
import os
import sys
import calendar
import numpy as np
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import matplotlib.ticker as mticker
from statsmodels.tsa.seasonal import STL

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import config
config.apply_plot_style()

In [ ]:
df = config.load_theft_data()   # one row per stolen-vehicle report (class 510)
print(f"Theft reports      : {len(df):,}")
print(f"Year range         : {int(df['year'].min())}-{int(df['year'].max())}")
print(f"Unknown time       : {df['time_unknown'].sum():,} ({df['time_unknown'].mean()*100:.1f}%)")

In [ ]:
# Monthly series
monthly = df.groupby(["year", "month"]).size().rename("count").reset_index()
monthly["period"] = pd.to_datetime(dict(year=monthly["year"], month=monthly["month"], day=1))
full = monthly.set_index("period")["count"]
monthly = (monthly.set_index("period").reindex(pd.date_range("2000-01-01", config.MONTHLY_SERIES_END, freq="MS"), fill_value=0)
           .rename_axis("period").reset_index())
# Incomplete end of the 2022 extract (see config.MONTHLY_SERIES_END): excluded from all monthly analyses
tail = full[full.index > pd.Timestamp(config.MONTHLY_SERIES_END)]
ref = full[(full.index >= "2022-01-01") & (full.index <= config.MONTHLY_SERIES_END)].mean()
print("Excluded incomplete months:", {f"{k:%Y-%m}": int(v) for k, v in tail.items()},
      f"(Jan-Oct 2022 mean {ref:.0f}/month)")
monthly["year"], monthly["month"] = monthly["period"].dt.year, monthly["period"].dt.month
monthly["days"] = monthly["period"].dt.days_in_month
monthly["daily_rate"] = monthly["count"] / monthly["days"]
print(f"Monthly series : {len(monthly)} months, counts {monthly['count'].min():,}-{monthly['count'].max():,}")
print(f"Min month: {monthly.loc[monthly['count'].idxmin(), 'period']:%Y-%m}  Max month: {monthly.loc[monthly['count'].idxmax(), 'period']:%Y-%m}")

#### STL Decomposition

STL is applied to the daily rate (count / days in month) so that month length does not enter the seasonal term.

In [ ]:
series = pd.Series(monthly["daily_rate"].values, index=monthly["period"])
result = STL(series, period=12, seasonal=13, robust=True).fit()
trend, seasonal, residual = result.trend, result.seasonal, result.resid
print(f"Seasonal amplitude (max-min) : {seasonal.max() - seasonal.min():.2f} thefts/day")
print(f"Trend range                  : {trend.min():.1f} - {trend.max():.1f} thefts/day")
print(f"Residual std                 : {residual.std():.2f}")

fig = result.plot()
fig.set_size_inches(*config.FIGURE_SIZE_FULL)
fig.suptitle("Vehicle Theft per Day (monthly) - STL Decomposition", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig4_monthly_stl_decomposition.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Change-point detection (PELT)

PELT is run on the **deseasonalised** series (observed minus STL seasonal), not on the already-smoothed STL
trend. Because the series trends rather than jumping between flat levels, the cost is a continuous
piecewise-linear model (`clinear`): change points are changes of slope. Penalty is BIC-type,
`pen = 2 params x sigma^2 x log(n)`, with sigma the STL residual standard deviation. Minimum segment 12 months.

In [ ]:
deseason = (series - seasonal).values
n_obs = len(deseason)
sigma = residual.std()
PENALTY = 2 * 2 * sigma ** 2 * np.log(n_obs)
algo = rpt.Pelt(model="clinear", min_size=12, jump=1).fit(deseason)
breakpoints = algo.predict(pen=PENALTY)
bp_indices = np.asarray(breakpoints[:-1], dtype=int)          # first index of each new segment
print(f"sigma = {sigma:.2f}, penalty = {PENALTY:.1f}, change-points: {len(bp_indices)}")

segments = np.split(np.arange(n_obs), bp_indices)
fits = [np.polyfit(s, deseason[s], 1) for s in segments]      # per-segment line: slope per month, intercept
cp_rows = []
for i, idx in enumerate(bp_indices):
    cp_rows.append({
        "change_point": monthly["period"].iloc[idx].strftime("%Y-%m"),
        "level_at_change_per_day": round(np.polyval(fits[i + 1], idx), 1),
        "slope_before_per_day_per_year": round(fits[i][0] * 12, 2),
        "slope_after_per_day_per_year": round(fits[i + 1][0] * 12, 2),
    })
table2 = pd.DataFrame(cp_rows)
table2.to_csv(os.path.join(config.TABLES_DIR, "table2_changepoints.csv"), index=False)
print("Table 2 - Change points in the slope of the deseasonalised daily theft rate")
print(table2.to_string(index=False))

# Sensitivity: number of change points across a range of penalties
sens = pd.DataFrame([{"penalty_multiplier": m, "n_changepoints": len(algo.predict(pen=PENALTY * m)) - 1}
                     for m in [0.5, 1, 2, 4]])
print(sens.to_string(index=False))

In [ ]:
x_num = np.arange(n_obs)
jan_idx = monthly.index[monthly["month"] == 1].tolist()
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
ax.bar(x_num, monthly["daily_rate"], width=1.0, color="#2166ac", alpha=0.85, zorder=2, label="Thefts per day (monthly)")
ax.plot(x_num, trend.values, color="#b2182b", linewidth=2.5, zorder=3, label="STL trend")
for s, f in zip(segments, fits):
    ax.plot(s, np.polyval(f, s), color="#333", linewidth=1.8, zorder=4,
            label="PELT piecewise-linear fit" if s[0] == 0 else None)
for idx in bp_indices:
    ax.axvline(x=idx, color="#b2182b", linewidth=1.5, linestyle="--", alpha=0.85, zorder=4)
ax.set_xticks(jan_idx)
ax.set_xticklabels(monthly.loc[jan_idx, "year"].astype(str), rotation=45, ha="right", fontsize=12)
ax.set_xlabel("Year", fontsize=18)
ax.set_ylabel("Thefts per day", fontsize=18)
ax.set_title("Vehicle Theft per Day with STL Trend and Change-Points", pad=12, fontsize=18)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=12, loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=3)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig5_monthly_stl_changepoints.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Monthly seasonality boxplot

In [ ]:
month_labels = list(calendar.month_abbr)[1:]
boxes = [monthly.loc[monthly["month"] == m, "daily_rate"].values for m in range(1, 13)]
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
bp = ax.boxplot(boxes, patch_artist=True, widths=0.55,
                medianprops=dict(color="#d6604d", linewidth=2),
                flierprops=dict(marker="o", markersize=4, alpha=0.4, markeredgecolor="#888"))
for patch in bp["boxes"]:
    patch.set_facecolor("#aec7e8")
    patch.set_alpha(0.75)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_xlabel("Month")
ax.set_ylabel("Thefts per day")
ax.set_title("Monthly Seasonality of Vehicle Theft (daily rate, 2000-2022)", pad=12)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig6_seasonality_boxplot.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

# Seasonal profile from the STL seasonal component (independent of the long-run trend)
seas_profile = pd.Series(seasonal.values, index=monthly["month"]).groupby(level=0).mean()
print("Mean STL seasonal effect by month (thefts/day):")
print(seas_profile.round(2).set_axis(month_labels).to_string())

#### Theft vs Temperature

Two tests, with both series treated the same way:
1. **Seasonal cycle**: STL seasonal effect per calendar month vs mean temperature per calendar month (n = 12).
2. **Anomalies**: STL residual (theft with trend and seasonality removed) vs temperature anomaly
   (monthly mean temperature minus that month's 2000-2022 climatology), one value per month of the series (n = 274, January 2000 to October 2022).

In [ ]:
weather = pd.read_csv(config.WEATHER_DAILY, parse_dates=["date"])
wm = weather.groupby(["year", "month"])["mean_temp_c"].mean().reset_index()
merged = monthly.merge(wm, on=["year", "month"], how="inner")
merged["stl_seasonal"] = seasonal.values
merged["stl_resid"] = residual.values
merged["temp_anom"] = merged["mean_temp_c"] - merged.groupby("month")["mean_temp_c"].transform("mean")

clim = merged.groupby("month").agg(temp=("mean_temp_c", "mean"), seas=("stl_seasonal", "mean"))
rho_c, p_c = spearmanr(clim["temp"], clim["seas"])
rho_a, p_a = spearmanr(merged["temp_anom"], merged["stl_resid"])
table14 = pd.DataFrame([
    {"test": "seasonal cycle (calendar-month means)", "n": len(clim), "spearman_rho": rho_c, "p_value": p_c},
    {"test": "anomalies (STL residual vs temperature anomaly)", "n": len(merged), "spearman_rho": rho_a, "p_value": p_a},
])
table14.to_csv(os.path.join(config.TABLES_DIR, "table14_temperature.csv"), index=False)
print(table14.round(4).to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=config.FIGURE_SIZE_WIDE)
ax1.scatter(clim["temp"], clim["seas"], s=60, color="#2166ac", zorder=3)
for m, r in clim.iterrows():
    ax1.annotate(month_labels[m - 1], (r["temp"], r["seas"]), xytext=(4, 4), textcoords="offset points", fontsize=10)
ax1.axhline(0, color="#888", lw=0.8)
ax1.set_xlabel("Mean monthly temperature (°C)")
ax1.set_ylabel("STL seasonal effect (thefts/day)")
ax1.set_title(f"Seasonal cycle: Spearman ρ = {rho_c:.2f} (p = {p_c:.3f}, n = 12)", fontsize=12)
ax2.scatter(merged["temp_anom"], merged["stl_resid"], s=18, alpha=0.6, color="#2166ac", edgecolors="none")
ax2.axhline(0, color="#888", lw=0.8)
ax2.axvline(0, color="#888", lw=0.8)
ax2.set_xlabel("Temperature anomaly (°C)")
ax2.set_ylabel("STL residual (thefts/day)")
ax2.set_title(f"Anomalies: Spearman ρ = {rho_a:.2f} (p = {p_a:.3f}, n = {len(merged)})", fontsize=12)
for a in (ax1, ax2):
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Vehicle Theft vs Temperature ({config.WEATHER_LAT}°N, {abs(config.WEATHER_LON)}°W)")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig7_theft_vs_temperature.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Hour × day-of-week heatmap

Only reports with a recorded occurrence time are used (2022 has none; '00:01' placeholders are excluded).

In [ ]:
DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
known = df[~df["time_unknown"]].dropna(subset=["hour", "dow"])
pivot = (known.groupby(["dow", "hour"]).size().unstack(fill_value=0)
         .reindex(index=range(7), columns=range(24), fill_value=0))
share = pivot / pivot.values.sum() * 100

fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
im = ax.imshow(share.values, aspect="auto", cmap="viridis", interpolation="nearest")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=90, fontsize=12)
ax.set_yticks(range(7))
ax.set_yticklabels(DOW_LABELS, fontsize=12)
ax.set_xlabel("Hour of day", fontsize=18)
ax.set_ylabel("Day of week", fontsize=18)
ax.set_title(f"Vehicle Theft by Hour and Day of Week (known times, n = {len(known):,})", pad=12, fontsize=16)
cbar = plt.colorbar(im, ax=ax, pad=0.02, shrink=0.85)
cbar.set_label("Share of thefts (%)")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig8_hour_dow_heatmap.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

hour_share = share.sum(axis=0)
print("Share by hour (%):")
print(hour_share.round(2).to_string())
print(f"Peak hour: {hour_share.idxmax()}:00 ; 18:00-23:59 share = {hour_share.loc[18:23].sum():.1f}% ; "
      f"04:00-07:59 share = {hour_share.loc[4:7].sum():.1f}%")